In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(performance)
  library(lme4)
})

In [ ]:
tbl1 <- read.csv("036_ADNI Merge Modelling Cohort")

### Variables to Screen for Outliers

In [ ]:
colnames(tbl1)

In [ ]:
# Variables to check:
    # IR Measures
    # BMI
    # AD Biomarkers
vars_to_check <- c("METSIR_z", "TYG_z", "TYG_BMI_z", "TG_HDL_z", "BMI_z", "ABETA_clean", "TAU", "PTAU_clean")

### Identifying Outliers

In [ ]:
outlier_result <- check_outliers(
  tbl1[vars_to_check],
  method = "zscore_robust"
)

outlier_result

In [ ]:
outlier_df <- as.data.frame(outlier_result)
head(outlier_df)

In [ ]:
# Counting the number of outliers
tbl1_flagged <- tbl1 |>
  mutate(
    row_id = seq_len(n()),
    outlier_flag_raw = outlier_df$Outlier,
    is_outlier = as.logical(outlier_flag_raw)
  )

tbl1_flagged |>
  count(is_outlier)

In [ ]:
# Inspecting Candidiate Outliers
candidate_outliers <- tbl1_flagged |>
  filter(is_outlier) |>
  select(row_id, RID, VISCODEnum_z, all_of(vars_to_check)) |>
  arrange(RID, VISCODEnum_z)

candidate_outliers

### Visual Checks

In [ ]:
ggplot(tbl1_flagged, aes(x = TAU)) +
  geom_histogram(bins = 40) +
  labs(
    title = "Distribution of biomarker before treatment",
    x = "TAU",
    y = "Count"
  )

In [ ]:
ggplot(tbl1_flagged, aes(x = "", y = TAU, colour = is_outlier)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(width = 0.12, alpha = 0.5) +
  labs(
    title = "TAU-IR values with outlier flags",
    x = NULL,
    y = "TAU"
  )

In [ ]:
ggplot(tbl1_flagged, aes(x = VISCODEnum_z, y = TAU, colour = is_outlier)) +
  geom_point(alpha = 0.6) +
  labs(
    title = "Biomarker over time with flagged rows highlighted",
    x = "Time",
    y = "TAU"
  )

### Capping 

In [ ]:
# Capping Function
cap_percentile <- function(x, probs = c(0.01, 0.99), na.rm = TRUE) {
  stopifnot(is.numeric(x), length(probs) == 2)
  qs <- quantile(x, probs = probs, na.rm = na.rm, names = FALSE)
  pmin(pmax(x, qs[1]), qs[2])
}

In [ ]:
tbl1_capped <- tbl1_flagged |>
  mutate(
    TAU_capped = cap_percentile(TAU, probs = c(0.01, 0.99)),
    ABETA_capped = cap_percentile(ABETA_clean, probs = c(0.01, 0.99)),
    PTAU_capped = cap_percentile(PTAU_clean, probs = c(0.01, 0.99)),
    METSIR_capped = cap_percentile(METSIR_z, probs = c(0.01, 0.99)),
    TYG_capped = cap_percentile(TYG_z, probs = c(0.01, 0.99)),
    TYG_BMI_capped = cap_percentile(TYG_BMI_z, probs = c(0.01, 0.99)),
    TG_HDL_capped = cap_percentile(TG_HDL_z, probs = c(0.01, 0.99)),
    BMI_capped = cap_percentile(BMI_z, probs = c(0.01, 0.99))
  )

### Comparing Before & After Capping

In [ ]:
comparison_tab <- tibble(
  variable = c("METSIR_z", "METSIR_capped", "TYG_z", "TYG_capped", "TYG_BMI_z", "TYG_BMI_capped", 
               "TG_HDL_z", "TG_HDL_capped", "BMI", "BMI_capped", 
               "TAU", "TAU_capped", "ABETA_clean", "ABETA_capped", "PTAU_clean", "PTAU_capped"),
  min = c(
    min(tbl1_capped$METSIR_z, na.rm = TRUE),
    min(tbl1_capped$METSIR_capped, na.rm = TRUE),
    min(tbl1_capped$TYG_z, na.rm = TRUE),
    min(tbl1_capped$TYG_capped, na.rm = TRUE),
    min(tbl1_capped$TYG_BMI_z, na.rm = TRUE),
    min(tbl1_capped$TYG_BMI_capped, na.rm = TRUE),
    min(tbl1_capped$TG_HDL_z, na.rm = TRUE),
    min(tbl1_capped$TG_HDL_capped, na.rm = TRUE),
    min(tbl1_capped$ABETA_clean, na.rm = TRUE),
    min(tbl1_capped$ABETA_capped, na.rm = TRUE),
    min(tbl1_capped$TAU, na.rm = TRUE),
    min(tbl1_capped$TAU_capped, na.rm = TRUE),
    min(tbl1_capped$PTAU_clean, na.rm = TRUE),
    min(tbl1_capped$PTAU_capped, na.rm = TRUE),
    min(tbl1_capped$BMI_z, na.rm = TRUE),
    min(tbl1_capped$BMI_capped, na.rm = TRUE)
  ),
  p01 = c(
    quantile(tbl1_capped$METSIR_z, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$METSIR_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TYG_z, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TYG_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TYG_BMI_z, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TYG_BMI_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TG_HDL_z,0.01, na.rm = TRUE),
    quantile(tbl1_capped$TG_HDL_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$ABETA_clean, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$ABETA_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TAU, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$TAU_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$PTAU_clean, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$PTAU_capped, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$BMI_z, 0.01, na.rm = TRUE),
    quantile(tbl1_capped$BMI_capped, 0.01, na.rm = TRUE)
  ),
  median = c(
    median(tbl1_capped$METSIR_z, na.rm = TRUE),
    median(tbl1_capped$METSIR_capped, na.rm = TRUE),
    median(tbl1_capped$TYG_z, na.rm = TRUE),
    median(tbl1_capped$TYG_capped, na.rm = TRUE),
    median(tbl1_capped$TYG_BMI_z, na.rm = TRUE),
    median(tbl1_capped$TYG_BMI_capped, na.rm = TRUE),
    median(tbl1_capped$TG_HDL_z, na.rm = TRUE),
    median(tbl1_capped$TG_HDL_capped, na.rm = TRUE),
    median(tbl1_capped$ABETA_clean, na.rm = TRUE),
    median(tbl1_capped$ABETA_capped, na.rm = TRUE),
    median(tbl1_capped$TAU, na.rm = TRUE),
    median(tbl1_capped$TAU_capped, na.rm = TRUE),
    median(tbl1_capped$PTAU_clean, na.rm = TRUE),
    median(tbl1_capped$PTAU_capped, na.rm = TRUE),
    median(tbl1_capped$BMI_z, na.rm = TRUE),
    median(tbl1_capped$BMI_capped, na.rm = TRUE)
  ),
  p99 = c(
    quantile(tbl1_capped$METSIR_z, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$METSIR_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TYG_z, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TYG_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TYG_BMI_z, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TYG_BMI_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TG_HDL_z,0.99, na.rm = TRUE),
    quantile(tbl1_capped$TG_HDL_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$ABETA_clean, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$ABETA_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TAU, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$TAU_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$PTAU_clean,0.99, na.rm = TRUE),
    quantile(tbl1_capped$PTAU_capped, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$BMI_z, 0.99, na.rm = TRUE),
    quantile(tbl1_capped$BMI_capped, 0.99, na.rm = TRUE)
  ),
  max = c(
    max(tbl1_capped$METSIR_z, na.rm = TRUE),
    max(tbl1_capped$METSIR_capped, na.rm = TRUE),
    max(tbl1_capped$TYG_z, na.rm = TRUE),
    max(tbl1_capped$TYG_capped, na.rm = TRUE),
    max(tbl1_capped$TYG_BMI_z, na.rm = TRUE),
    max(tbl1_capped$TYG_BMI_capped, na.rm = TRUE),
    max(tbl1_capped$TG_HDL_z, na.rm = TRUE),
    max(tbl1_capped$TG_HDL_capped, na.rm = TRUE),
    max(tbl1_capped$ABETA_clean, na.rm = TRUE),
    max(tbl1_capped$ABETA_capped, na.rm = TRUE),
    max(tbl1_capped$TAU, na.rm = TRUE),
    max(tbl1_capped$TAU_capped, na.rm = TRUE),
    max(tbl1_capped$PTAU_clean, na.rm = TRUE),
    max(tbl1_capped$PTAU_capped, na.rm = TRUE),
    max(tbl1_capped$BMI_z, na.rm = TRUE),
    max(tbl1_capped$BMI_capped, na.rm = TRUE)
  )
)

comparison_tab

In [ ]:
fit_original <- lmer(
  ABETA_clean ~ VISCODEnum_z * METSIR_z + AGE_z + BMI_z + APOE4 + PTGENDER + PTEDUCAT_z + DIAGNOSIS_bl + (VISCODEnum_z | RID),
  data = tbl1_capped,
  REML = FALSE
)

fit_capped <- lmer(
  ABETA_capped ~ VISCODEnum_z * METSIR_capped + AGE_z + BMI_capped + APOE4 + PTGENDER + PTEDUCAT_z + DIAGNOSIS_bl + (VISCODEnum_z | RID),
  data = tbl1_capped,
  REML = FALSE
)

coef_tab <- tibble(
  term = names(fixef(fit_original)),
  estimate_original = unname(fixef(fit_original)),
  estimate_capped = unname(fixef(fit_capped)),
  delta = estimate_capped - estimate_original
)

coef_tab

In [ ]:
summary(fit_capped)

In [ ]:
summary(fit_original)

In [ ]:
colnames(tbl1_capped)

In [ ]:
write.csv(tbl1_capped, "050_ADNI_Merge_Cohort_V3")